# Notebook 5-6 — Analyse spatiale complete et robustesse

**Objectif :** Autocorrelation spatiale (Moran's I global, LISA) et Moran's I bivarie (mutations NF x proximite Seveso SH). Tests de sensibilite par rayons de voisinage (1-10 km) et sous-groupes.

---

## Donnees en entree

- `../data/1_patients/patients_geocoded_clean_idf_2018_2023.csv` — cohorte geocodee IDF
- `../data/3_analyses/patients_clinics_air_metrics.csv` — table d'analyse avec expositions
- `../data/0_brut/geospatial/icpe/icpe.geojson/icpe_idf.shp` — sites ICPE Seveso

## Donnees en sortie

- `../figures/resultats_moran/` — scatter Moran, permutations, cartes LISA
- `../data/3_analyses/` — synthese_moran.csv, patients_moran.csv

---

> **RGPD** : Ce notebook traite des donnees de sante pseudonymisees.
> Les fichiers `data/4_confidentiel/` ne doivent jamais etre versionnés sur git.
> **Chemin racine :** `h:/PFE Loice/Notebooks/Loice_Canc-air_2025/loice_pneumodetect/`

# LUNG-CANC'AIR — Analyse spatiale de Moran & Tests de robustesse

> **Objectif :** Tester l'association spatiale entre la présence de mutations NF
> (non-fumeurs) et la proximité des industries Seveso Seuil Haut (SH) en IDF —
> **puis vérifier si ce résultat tient la route** une fois mis à l'épreuve.

Ce notebook fusionne en une seule analyse cohérente :
- **l'analyse principale** (Parties 0 à 7) : la question posée, la méthode, les résultats bruts ;
- **les tests de robustesse** (Parties 8 à 15) : six vérifications qui remettent en cause
  le résultat principal avant toute conclusion.

Chaque calcul n'est fait **qu'une seule fois** ; les parties de robustesse réutilisent les
objets déjà calculés plus haut (`df`, `W`, `res_moran_*`, `res_bv_*`, `clusters_bv_A`...)
plutôt que de les recalculer.

| # | Partie | Rôle |
|---|---|---|
| 0 | Configuration | Paramètres et chemins (un seul jeu, fusionné) |
| 1 | Chargement des données | Patients, ICPE/Seveso, expositions |
| 2 | Matrice de poids spatiaux | Voisinage à 3 km |
| 3 | Moran's I global univarié | Les mutations NF se regroupent-elles entre elles ? |
| 4 | LISA local univarié | Où, sur la carte ? |
| 5 | Moran's I bivarié ★ | **Question centrale** : NF × Seveso SH |
| 6 | LISA bivarié ★ | Où sont les co-clusters ? |
| 7 | Synthèse & verdict provisoire | Premier bilan — à mettre à l'épreuve |
| 8 | Robustesse — Densité (ville/campagne) | La densité explique-t-elle le résultat ? |
| 9 | Robustesse — Sensibilité au rayon | Le résultat dépend-il du choix de 3 km ? |
| 10 | Robustesse — Concentration géographique | Sites Seveso & patients du co-cluster |
| 11 | Robustesse — Tests multiples | A-t-on testé trop de choses ? |
| 12 | Robustesse — Mutation par mutation | Le signal est-il partagé par toutes ? |
| 13 | Robustesse — ROS1/ERBB2/MET vs EGFR/ALK | Comparaison groupée formelle |
| 14 | Cohérence avec l'analyse individuelle | Convergence avec la régression logistique |
| 15 | Synthèse finale globale | Verdict d'ensemble |

**Script utilisé :** `moran_spatial_analysis.py`

---
## PARTIE 0 — Configuration

> Modifier ici les chemins et paramètres. Ne rien changer en dessous.
> **Un seul jeu de paramètres** pour toute l'analyse (principale + robustesse).

In [ ]:
import sys
import importlib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import geopandas as gpd

# Ajouter le dossier du script au path
sys.path.append(r"H:\PFE Loice\Notebooks\Loice_Canc-air_2025\loice_pneumodetect\pneumodetect\src")

import moran_spatial_analysis as moran
importlib.reload(moran)

# =============================================================================
# ★ CONFIGURATION — Modifier ici uniquement ★
# =============================================================================
CONFIG = {

    # ── Chemins ───────────────────────────────────────────────────────────────
    'patients_csv' : r"../data/1_patients/patients_geocoded_clean_idf_2018_2023.csv",
    'icpe_shp'     : r"../data/0_brut/geospatial/icpe/icpe.geojson/icpe_idf.shp",
    'output_dir'   : r"H:\PFE Loice\Notebooks\Loice_Canc-air_2025\loice_pneumodetect\Notebooks\output\resultats_moran",

    # ── IRIS : géométrie (shp) + population (csv) — pour la Partie 8 (densité) ──
    # Deux fichiers séparés à assembler (cf. cellule de diagnostic en Partie 8).
    'iris_shp'         : r"À_COMPLETER\contours_iris_idf.shp",
    'iris_pop_csv'     : r"À_COMPLETER\population_iris_idf.csv",
    'iris_csv_sep'     : ';',        # les fichiers INSEE sont souvent en ';'
    'iris_csv_encoding': 'latin1',   # et en latin1/cp1252, pas en utf-8
    'col_pop'          : 'P21_POP',     # ← colonne "Population" totale (à confirmer)
    'col_iris_code_csv': 'IRIS',        # ← colonne code IRIS du csv (à confirmer)
    'col_iris_code_shp': 'CODE_IRIS',   # ← colonne code IRIS du shapefile (confirmé : présent aussi dans le fichier patients)

    # ── Paramètres spatiaux ───────────────────────────────────────────────────
    'rayon_m'        : 3000,    # ← rayon de voisinage en mètres
    'n_permutations' : 999,     # ← nb permutations pour les tests
    'alpha'          : 0.05,    # ← seuil de significativité
    'rayons_test_m'  : [1000, 2000, 3000, 5000, 10000],   # ← Partie 9 (sensibilité au rayon)

    # ── Colonnes patients ─────────────────────────────────────────────────────
    'col_x'      : 'x',                # longitude WGS84
    'col_y'      : 'y',                # latitude WGS84
    'col_id'     : 'pseudo_provisoire',
    'col_tabac'  : 'paquet_annee',
    'col_age'    : 'age_diagnostic',   # ← Partie 8 (régression ajustée)
    'col_sexe'   : 'sexe',             # ← Partie 8 (régression ajustée)

    # ── Mutations NF à considérer ─────────────────────────────────────────────
    'mutations_nf' : ['EGFR', 'ALK', 'ROS1', 'RET', 'NTRK', 'ERBB2', 'MET'],
}

print("✅ Configuration définie")
print(f"   Rayon voisinage  : {CONFIG['rayon_m']} m")
print(f"   Permutations     : {CONFIG['n_permutations']}")
print(f"   Mutations NF     : {CONFIG['mutations_nf']}")
print(f"   Figures sauvées  : {CONFIG['output_dir']}")


In [ ]:
# Fonction utilitaire partagée (utilisée dans les Parties 3, 10, 12, 13) :
# convertit une colonne de mutation en 0/1, quel que soit son encodage d'origine.
def colonne_mutation_en_binaire(serie):
    """0/1 quel que soit l'encodage (déjà 0/1, 'Oui'/'Non', nom de la mutation en texte
    quand présente, NaN quand absente, ...)."""
    if pd.api.types.is_numeric_dtype(serie) or pd.api.types.is_bool_dtype(serie):
        return serie.fillna(0).astype(int)
    s = serie.astype(str).str.strip().str.lower()
    valeurs_absence = {'', 'nan', 'non', 'no', 'false', '0', 'absent', 'negatif', 'négatif', 'none', 'na'}
    return (~s.isin(valeurs_absence)).astype(int)

print("✅ Fonction colonne_mutation_en_binaire() prête")


---
## PARTIE 1 — Chargement des données

> Charge les patients géocodés, crée les groupes (mutations NF, non-fumeurs),
> projette en Lambert 93 et calcule les variables d'exposition ICPE.

In [ ]:
# ── Chargement patients ──────────────────────────────────────────────────────
df, gdf = moran.charger_patients(CONFIG)

# ── Chargement ICPE ──────────────────────────────────────────────────────────
icpe, icpe_sh, icpe_sb, icpe_ns = moran.charger_icpe(CONFIG)

# ── Calcul expositions ICPE par patient ──────────────────────────────────────
df = moran.calculer_expositions_icpe(df, icpe_sh, icpe_sb, CONFIG['rayon_m'])

print(f"\n✅ Données prêtes")
print(f"   Patients           : {len(df)}")
print(f"   Groupe A (mut. NF) : {df['groupe_A'].sum()} ({df['groupe_A'].mean()*100:.1f}%)")
print(f"   Groupe C (non-fum) : {df['groupe_C'].sum()} ({df['groupe_C'].mean()*100:.1f}%)")
print(f"   Groupe A+C         : {df['groupe_AC'].sum()} ({df['groupe_AC'].mean()*100:.1f}%)")
print(f"   Seveso SH          : {len(icpe_sh)} sites")
print(f"   dist SH médiane    : {df['dist_SH_m'].median():.0f} m")
print(f"   nb SH 3km médiane  : {df['nb_SH_3km'].median():.0f}")


#### 📊 Résultats — Chargement données

✅ **1682 patients** chargés. Groupe A (mutations NF) = **332 (19,7%)**, Groupe C
(non-fumeurs) = **241 (14,3%)**, Groupe A+C = **434 (25,8%)**. **37 sites Seveso SH**
identifiés en IDF (sur 2919 ICPE au total). Distance médiane à un site SH : **7727 m** ;
nombre médian de sites SH dans 3 km : **0** — l'exposition Seveso SH est rare et
concentrée sur une minorité de patients, ce qui sera important à garder en tête pour
la suite (Partie 10).

### 1b — Statistiques descriptives spatiales

In [ ]:
# Comparaison exposition SH : Groupe A vs Groupe B
from scipy.stats import mannwhitneyu

g_A = df[df['groupe_A']==1]['dist_SH_m'].values
g_B = df[df['groupe_A']==0]['dist_SH_m'].values
_, p_dist = mannwhitneyu(g_A, g_B, alternative='two-sided')

g_A_nb = df[df['groupe_A']==1]['nb_SH_3km'].values
g_B_nb = df[df['groupe_A']==0]['nb_SH_3km'].values
_, p_nb = mannwhitneyu(g_A_nb, g_B_nb, alternative='two-sided')

print("── Exposition Seveso SH : Groupe A (mutations NF) vs Groupe B ──")
print(f"  dist_SH_m  : moyenne A={g_A.mean():.0f}m | B={g_B.mean():.0f}m | "
      f"p={'<0.001' if p_dist<0.001 else f'{p_dist:.3f}'} "
      f"{'✅' if p_dist<0.05 else '—'}")
print(f"  nb_SH_3km  : moyenne A={g_A_nb.mean():.2f} | B={g_B_nb.mean():.2f} | "
      f"p={'<0.001' if p_nb<0.001 else f'{p_nb:.3f}'} "
      f"{'✅' if p_nb<0.05 else '—'}")

# Comparaison Groupe C vs D
g_C = df[df['groupe_C']==1]['dist_SH_m'].values
g_D = df[df['groupe_C']==0]['dist_SH_m'].values
_, p_dist_cd = mannwhitneyu(g_C, g_D, alternative='two-sided')

print(f"\n── Exposition Seveso SH : Groupe C (non-fumeurs) vs Groupe D ──")
print(f"  dist_SH_m  : moyenne C={g_C.mean():.0f}m | D={g_D.mean():.0f}m | "
      f"p={'<0.001' if p_dist_cd<0.001 else f'{p_dist_cd:.3f}'} "
      f"{'✅' if p_dist_cd<0.05 else '—'}")

# Boxplots
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle("Exposition Seveso SH par groupe", fontsize=13, fontweight='bold')

for ax, (var, lbl) in zip(axes,
    [('dist_SH_m','Distance SH (m)'),('nb_SH_3km','Nb SH 3km'),
     ('score_expo_SH','Score expo SH')]):
    data_plot = [df[df['groupe_A']==1][var].values,
                 df[df['groupe_A']==0][var].values]
    bp = ax.boxplot(data_plot, labels=['Groupe A\n(mut. NF)','Groupe B'],
                    patch_artist=True, widths=0.5)
    bp['boxes'][0].set_facecolor('#2E86AB'); bp['boxes'][0].set_alpha(0.7)
    bp['boxes'][1].set_facecolor('#AAAAAA'); bp['boxes'][1].set_alpha(0.7)
    ax.set_ylabel(lbl); ax.set_title(lbl, fontweight='bold')
    ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout(); plt.show()


#### 📊 Résultats — Statistiques descriptives spatiales

Aucune différence significative entre le Groupe A et le reste : `dist_SH_m` (p=0,818),
`nb_SH_3km` (p=0,113) ; même chose pour Groupe C vs D (p=0,402). **Au niveau individuel
simple — sans tenir compte de la géographie — les patients NF ne semblent pas
systématiquement plus proches des sites Seveso SH.** C'est précisément pour ça qu'on
passe à l'analyse spatiale : un effet peut exister sous forme de clusters géographiques
sans apparaître dans une simple comparaison de moyennes globales.

---
## PARTIE 2 — Matrice de poids spatiaux

> Construit la matrice W par rayon fixe (`rayon_m` dans CONFIG).
> Vérifie la connectivité — si trop de patients isolés, augmenter le rayon.

In [ ]:
W, n_voisins = moran.construire_matrice_poids(
    df,
    rayon_m = CONFIG['rayon_m'],
    n_permutations = CONFIG['n_permutations']
)

print(f"\n✅ Matrice W construite")
print(f"   Connectivité : {sum(1 for n in n_voisins if n>0)}/{len(df)} patients connectés")
print(f"   Patients isolés : {sum(1 for n in n_voisins if n==0)}")


#### 📊 Résultats — Matrice de poids spatiaux

Rayon de 3 km : médiane de **73 voisins** par patient (de 0 à 284) — **71 patients
isolés (4,2%)**, exclus des calculs de Moran ; **1611/1682 patients connectés**.

---
## PARTIE 3 — Moran's I global univarié

> Teste si chaque variable est spatialement autocorrélée.
>
> **I > 0 et p < 0.05** → les valeurs similaires sont regroupées géographiquement
> (clustering spatial).
>
> **I ≈ 0** → distribution aléatoire dans l'espace.

### 3a — Mutations NF (Groupe A)

In [ ]:
res_moran_A = moran.moran_global_univarie(
    df, W,
    variable = 'groupe_A',
    label    = 'Mutations NF (Groupe A)',
    couleur  = '#2E86AB',
    n_perm   = CONFIG['n_permutations']
)


#### 📊 Résultats — Moran I — Mutations NF

I = -0,0081, p = 0,224 → **pas de clustering significatif**. Les mutations NF sont
distribuées aléatoirement dans l'espace : les patients mutés NF ne se regroupent pas
entre eux plus que ne le voudrait le hasard.

### 3b — Non-fumeurs (Groupe C)

In [ ]:
res_moran_C = moran.moran_global_univarie(
    df, W,
    variable = 'groupe_C',
    label    = 'Non-fumeurs (Groupe C)',
    couleur  = '#52B788',
    n_perm   = CONFIG['n_permutations']
)


#### 📊 Résultats — Moran I — Non-fumeurs

I = 0,0010, p = 0,370 → **pas de clustering significatif** non plus pour les
non-fumeurs : même conclusion que pour le Groupe A.

### 3c — Score exposition Seveso SH

In [ ]:
res_moran_SH = moran.moran_global_univarie(
    df, W,
    variable = 'score_expo_SH',
    label    = 'Score exposition Seveso SH',
    couleur  = '#E76F51',
    n_perm   = CONFIG['n_permutations']
)


#### 📊 Résultats — Moran I — Score exposition SH

I = 0,6563, p = 0,001 → **fort clustering significatif**. Résultat attendu : les sites
industriels sont par nature concentrés géographiquement (vallées industrielles, zones
portuaires...), pas dispersés au hasard.

### 3d — Nombre de Seveso SH dans 3km

In [ ]:
res_moran_nb = moran.moran_global_univarie(
    df, W,
    variable = 'nb_SH_3km',
    label    = 'Nb Seveso SH dans 3km',
    couleur  = '#7B2D8B',
    n_perm   = CONFIG['n_permutations']
)


#### 📊 Résultats — Moran I — Nb SH 3km

I = 0,4877, p = 0,001 → fort clustering également, cohérent avec 3c (les deux mesurent
la même réalité : la concentration géographique des sites Seveso).

---
## PARTIE 4 — LISA (Indicateurs locaux d'association spatiale)

> Identifie **où** dans l'IDF se trouvent les clusters spatiaux.
>
> | Quadrant | Signification |
> |---|---|
> | **HH** (rouge) | Forte valeur entourée de fortes valeurs → cluster positif |
> | **LL** (vert) | Faible valeur entourée de faibles valeurs → cluster négatif |
> | **HL** (orange) | Forte valeur entourée de faibles valeurs → outlier |
> | **LH** (bleu) | Faible valeur entourée de fortes valeurs → outlier |

### 4a — LISA mutations NF (Groupe A)

In [ ]:
df, lisa_A, clusters_A = moran.lisa_local(
    df, W,
    variable = 'groupe_A',
    label    = 'Mutations NF (Groupe A)',
    icpe_sh  = icpe_sh,
    n_perm   = CONFIG['n_permutations'],
    alpha    = CONFIG['alpha']
)

print(f"\nClusters HH (mutations NF regroupées) : {clusters_A['HH']}")
print(f"→ Ces {clusters_A['HH']} patients forment des zones de concentration")
print(f"  de mutations NF. Coïncident-ils avec des Seveso SH ? → voir carte")


#### 📊 Résultats — LISA — Mutations NF

**1 seul patient** en cluster HH — confirme l'absence de clustering observée en 3a :
il n'y a quasiment aucune zone où les mutés NF se regroupent entre eux.

### 4b — LISA non-fumeurs (Groupe C)

In [ ]:
df, lisa_C, clusters_C = moran.lisa_local(
    df, W,
    variable = 'groupe_C',
    label    = 'Non-fumeurs (Groupe C)',
    icpe_sh  = icpe_sh,
    n_perm   = CONFIG['n_permutations'],
    alpha    = CONFIG['alpha']
)


#### 📊 Résultats — LISA — Non-fumeurs

**11 patients** en cluster HH — léger mais cohérent avec le Moran global quasi-nul
observé en 3b.

---
## PARTIE 5 — Moran's I bivarié ★

> **C'est la réponse directe à la question principale :**
> les zones avec beaucoup de patients mutés NF sont-elles
> spatialement associées aux zones proches des Seveso SH ?
>
> **I_biv > 0 et p < 0.05** → co-clustering positif :
> mutations NF et Seveso SH se retrouvent dans les mêmes zones.
>
> **I_biv ≈ 0** → aucune association spatiale entre les deux.

### 5a — Mutations NF × Score exposition SH

In [ ]:
res_bv_A_score = moran.moran_bivarie(
    df, W,
    var_x   = 'groupe_A',
    var_y   = 'score_expo_SH',
    label_x = 'Mutations NF',
    label_y = 'Score expo Seveso SH',
    n_perm  = CONFIG['n_permutations']
)

print(f"\nInterprétation :")
if res_bv_A_score['p_sim'] < CONFIG['alpha']:
    if res_bv_A_score['I'] > 0:
        print(f"  ✅ Association spatiale POSITIVE significative")
        print(f"     Les zones avec beaucoup de mutés NF sont proches des Seveso SH")
    else:
        print(f"  ✅ Association spatiale NÉGATIVE significative")
        print(f"     Les zones avec beaucoup de mutés NF s'éloignent des Seveso SH")
else:
    print(f"  — Pas d'association spatiale significative")
    print(f"     Les mutations NF sont distribuées indépendamment des Seveso SH")


#### 📊 Résultats — Moran bivarié — Mutations NF × Score expo SH

**I = 0,0164, p = 0,023 → co-clustering positif significatif, mais faible.** Les zones à
forte densité de mutés NF sont statistiquement associées aux zones proches des Seveso
SH. **C'est le résultat central de cette analyse — et celui que la suite du notebook
(Parties 8 à 15) va mettre à l'épreuve.** Notez la taille d'effet : I=0,016 est très
petit comparé au I=0,656 du clustering propre des sites Seveso (3c) — significatif ne
veut pas dire fort.

### 5b — Mutations NF × Nombre de Seveso SH dans 3km

In [ ]:
res_bv_A_nb = moran.moran_bivarie(
    df, W,
    var_x   = 'groupe_A',
    var_y   = 'nb_SH_3km',
    label_x = 'Mutations NF',
    label_y = 'Nb SH 3km',
    n_perm  = CONFIG['n_permutations']
)


#### 📊 Résultats — Moran bivarié — Mutations NF × Nb SH

I = 0,0227, p = 0,004 → significatif également, un peu plus net que 5a. Les deux
versions (score continu / comptage) de l'exposition Seveso pointent dans le même sens.

### 5c — Groupe A+C (mutations NF + non-fumeurs) × Score exposition SH

In [ ]:
res_bv_AC_score = moran.moran_bivarie(
    df, W,
    var_x   = 'groupe_AC',
    var_y   = 'score_expo_SH',
    label_x = 'Mutations NF + Non-fumeurs (A+C)',
    label_y = 'Score expo Seveso SH',
    n_perm  = CONFIG['n_permutations']
)


#### 📊 Résultats — Moran bivarié — Groupe A+C × Score expo SH

I = 0,0054, p = 0,208 → **plus significatif** une fois le groupe élargi aux non-fumeurs
sans mutation driver. ⚠️ Premier signe de fragilité : le résultat dépend de la
définition exacte du groupe testé — à creuser en Partie 12.

---
## PARTIE 6 — LISA bivarié ★

> Identifie **localement** les zones de co-clustering entre mutations NF et Seveso SH.
>
> **Quadrant HH** (rouge) → zones où patients mutés NF ET forte exposition SH coexistent.
> C'est ici qu'un signal géographique potentiellement important se manifeste.

### 6a — LISA bivarié : Mutations NF × Score exposition SH

In [ ]:
df, lisa_bv_A, clusters_bv_A = moran.lisa_bivarie(
    df, W,
    var_x   = 'groupe_A',
    var_y   = 'score_expo_SH',
    label_x = 'Mutations NF',
    label_y = 'Score expo SH',
    icpe_sh = icpe_sh,
    n_perm  = CONFIG['n_permutations'],
    alpha   = CONFIG['alpha']
)

print(f"\nCo-clusters HH (mutations NF ET forte expo SH) : {clusters_bv_A['HH']}")
if clusters_bv_A['HH'] > 0:
    # Localisation des clusters HH
    hh_patients = df[df['lisa_bv_type'] == 'HH'][['pseudo_provisoire',
                                                     'x_l93','y_l93',
                                                     'dist_SH_m','nb_SH_3km']]
    print(f"\nCaractéristiques des {clusters_bv_A['HH']} patients HH :")
    print(f"  dist_SH_m médiane  : {hh_patients['dist_SH_m'].median():.0f} m")
    print(f"  nb_SH_3km médiane  : {hh_patients['nb_SH_3km'].median():.1f}")
else:
    print("  → Aucun co-cluster HH significatif")

# ⚠️ CORRECTIF : moran.lisa_bivarie() écrit toujours dans une colonne nommée
# 'lisa_bv_type' (nom fixe), quel que soit var_x. L'appel suivant (6b, groupe_AC)
# écrirait dans la MÊME colonne et écraserait ce résultat. On la renomme donc
# immédiatement pour la préserver -- c'est ELLE qu'il faut utiliser en Partie 10,
# pas 'lisa_bv_type' (qui après la Partie 6b contiendra les résultats de 6b, pas 6a).
df = df.rename(columns={'lisa_bv_type': 'lisa_bv_type_A'})
print(f"\n✅ Colonne préservée sous 'lisa_bv_type_A' ({(df['lisa_bv_type_A']=='HH').sum()} HH) -- utilisée en Partie 10")


#### 📊 Résultats — LISA bivarié — Mutations NF × Score expo SH

**64 patients en co-cluster HH** (mutation NF + forte exposition SH). Distance médiane
de ces patients au site SH le plus proche : 3131 m ; nombre médian de sites dans 3 km :
0. ⚠️ Ce dernier chiffre peut sembler contre-intuitif pour des patients censés être
"fortement exposés" — le LISA bivarié classe un patient sur **sa valeur propre +
la moyenne de ses voisins**, pas uniquement sa propre exposition directe. On creuse
ces 64 patients en détail en **Partie 10**.

### 6b — LISA bivarié : Groupe A+C × Score exposition SH

In [ ]:
df, lisa_bv_AC, clusters_bv_AC = moran.lisa_bivarie(
    df, W,
    var_x   = 'groupe_AC',
    var_y   = 'score_expo_SH',
    label_x = 'Groupe A+C',
    label_y = 'Score expo SH',
    icpe_sh = icpe_sh,
    n_perm  = CONFIG['n_permutations'],
    alpha   = CONFIG['alpha']
)

# Même précaution qu'en 6a : on isole cette colonne sous son propre nom
# pour ne pas la perdre si une autre analyse LISA bivariée est ajoutée plus tard.
df = df.rename(columns={'lisa_bv_type': 'lisa_bv_type_AC'})


#### 📊 Résultats — LISA bivarié — Groupe A+C × Score expo SH

78 patients en co-cluster HH une fois le groupe élargi à A+C — cohérent avec la
dilution du signal déjà vue en 5c.

---
## PARTIE 7 — Synthèse des résultats & verdict provisoire

> Tableau récapitulatif de tous les indices de Moran calculés.
> Les résultats sont sauvegardés en CSV dans `output_dir`.

In [ ]:
resultats_global  = [res_moran_A, res_moran_C, res_moran_SH, res_moran_nb]
resultats_bivarie = [res_bv_A_score, res_bv_A_nb, res_bv_AC_score]

df_synthese = moran.synthese_resultats(resultats_global, resultats_bivarie)


#### 📊 Résultats — Synthèse Moran

Tableau récapitulatif des 7 tests : 2 univariés non significatifs (mutations NF,
non-fumeurs), 2 univariés très significatifs mais attendus (clustering propre des
sites Seveso), et 3 bivariés dont 2 significatifs (5a, 5b) et 1 non significatif (5c).

### 7b — Interprétation finale

In [ ]:
print("═"*65)
print("INTERPRÉTATION FINALE")
print("═"*65)

# Moran global groupe A
I_A = res_moran_A['I']; p_A = res_moran_A['p_sim']
print(f"\n1. Clustering spatial des mutations NF :")
print(f"   Moran I = {I_A:.4f} | p = {'<0.001' if p_A<0.001 else f'{p_A:.3f}'}")
if p_A < CONFIG['alpha']:
    if I_A > 0:
        print(f"   ✅ Les mutations NF sont spatialement CLUSTÉRISÉES en IDF")
        print(f"      (elles ne sont pas distribuées aléatoirement)")
    else:
        print(f"   ✅ Les mutations NF sont spatialement DISPERSÉES")
else:
    print(f"   — Les mutations NF sont distribuées ALÉATOIREMENT dans l'espace")

# Moran bivarié A × SH
I_bv = res_bv_A_score['I']; p_bv = res_bv_A_score['p_sim']
print(f"\n2. Association spatiale mutations NF × Seveso SH :")
print(f"   Moran I bivarié = {I_bv:.4f} | p = {'<0.001' if p_bv<0.001 else f'{p_bv:.3f}'}")
if p_bv < CONFIG['alpha']:
    if I_bv > 0:
        print(f"   ✅ CO-CLUSTERING POSITIF significatif")
        print(f"      Les zones à forte densité de mutés NF sont proches des Seveso SH")
        print(f"      → Signal spatial à investiguer")
    else:
        print(f"   ✅ CO-CLUSTERING NÉGATIF significatif")
        print(f"      Les mutés NF habitent LOIN des Seveso SH")
else:
    print(f"   — AUCUNE association spatiale significative")
    print(f"      Les mutations NF et les Seveso SH sont indépendants spatialement")

# Co-clusters HH
print(f"\n3. Co-clusters HH (mutations NF ET forte expo SH) :")
print(f"   Nombre de patients : {clusters_bv_A['HH']}")
if clusters_bv_A['HH'] > 0:
    print(f"   ⚠️  Ces zones méritent une investigation géospatiale approfondie")
else:
    print(f"   — Aucune zone de co-clustering détectée")

print(f"\n4. Mise en garde :")
print(f"   Un Moran bivarié significatif indique une co-localisation spatiale,")
print(f"   PAS une relation causale entre Seveso SH et mutations NF.")
print(f"   Des facteurs de confusion géographiques doivent être exclus.")


#### 📊 Résultats — Interprétation finale & verdict provisoire

**Verdict provisoire de l'analyse principale :** un signal positif faible mais
statistiquement détecté entre mutations NF et proximité des sites Seveso SH
(I=0,016, p=0,023 ; 64 patients en co-cluster). C'est un résultat **« significatif »,
pas forcément « solide »**.

---
## ⚠️ Avant de conclure : 6 vérifications de robustesse

Un résultat significatif sur un seul test, avec un seul rayon, sur un seul découpage de
groupe, peut être un coup de chance statistique. Les Parties 8 à 15 ci-dessous reprennent
**les objets déjà calculés plus haut** (`df`, `W`, `res_moran_*`, `res_bv_*`,
`clusters_bv_A`...) — rien n'est recalculé depuis zéro — pour tester si ce verdict
provisoire résiste à l'examen.

---
## PARTIE 8 — Robustesse : est-ce juste un effet "ville vs campagne" ?

**L'idée en clair :** les usines Seveso sont presque toujours construites loin des
centres-villes. Si les patients NF habitent aussi, par ailleurs, plutôt en zone moins
peuplée, le résultat de la Partie 5 ne dirait peut-être rien sur la pollution : il
dirait juste "les gens qui habitent loin de la ville habitent loin de la ville".

**Ce qu'on fait ici :**
1. On assemble vos deux fichiers IRIS : le shapefile (contours) + le csv (population), reliés par le code IRIS.
2. On calcule la densité = population / surface pour chaque IRIS, puis on l'attribue à chaque patient.
3. On regarde si les patients NF habitent des quartiers moins denses que les autres.
4. On refait le test de Moran bivarié de la Partie 5 **séparément dans des quartiers de densité comparable**.
5. On ajoute une régression logistique simple qui met toutes les variables ensemble.

In [ ]:
# ── Diagnostic : vérifier les vrais noms de colonnes avant de continuer ──
pop_diag = pd.read_csv(CONFIG['iris_pop_csv'], sep=CONFIG['iris_csv_sep'],
                       encoding=CONFIG['iris_csv_encoding'], nrows=5)
print("Colonnes du CSV population :")
print(pop_diag.columns.tolist())
print()
print(pop_diag.head())

iris_diag = gpd.read_file(CONFIG['iris_shp'], rows=5)
print("\nColonnes du shapefile IRIS :")
print(iris_diag.columns.tolist())

print('''
→ Repérez dans la liste ci-dessus :
   - la colonne "Population" totale (PAS une tranche d'âge) -> CONFIG['col_pop']
   - la colonne code IRIS du csv                            -> CONFIG['col_iris_code_csv']
   - la colonne code IRIS du shapefile                      -> CONFIG['col_iris_code_shp']
  Corrigez CONFIG en Partie 0 si les noms affichés ne correspondent pas.
''')


In [ ]:
# ── Chargement et assemblage : géométrie (shp) + population (csv) ───────────
try:
    iris_geo = gpd.read_file(CONFIG['iris_shp'])
    iris_geo = iris_geo.to_crs(epsg=2154)  # Lambert 93, même CRS que les patients

    iris_pop = pd.read_csv(CONFIG['iris_pop_csv'], sep=CONFIG['iris_csv_sep'],
                            encoding=CONFIG['iris_csv_encoding'])

    iris_geo[CONFIG['col_iris_code_shp']] = iris_geo[CONFIG['col_iris_code_shp']].astype(str).str.zfill(9)
    iris_pop[CONFIG['col_iris_code_csv']] = iris_pop[CONFIG['col_iris_code_csv']].astype(str).str.zfill(9)

    iris = iris_geo.merge(
        iris_pop[[CONFIG['col_iris_code_csv'], CONFIG['col_pop']]],
        left_on=CONFIG['col_iris_code_shp'], right_on=CONFIG['col_iris_code_csv'],
        how='left'
    )

    n_sans_pop = iris[CONFIG['col_pop']].isna().sum()
    print(f"✅ Jointure géométrie + population : {len(iris) - n_sans_pop}/{len(iris)} IRIS avec une population")
    if n_sans_pop > 0:
        print(f"   ⚠️  {n_sans_pop} IRIS sans population assignée (codes qui ne matchent pas -> à vérifier)")

    iris['surface_km2'] = iris.geometry.area / 1e6
    iris['densite_pop'] = iris[CONFIG['col_pop']] / iris['surface_km2']

    print(f"   Densité médiane : {iris['densite_pop'].median():.0f} hab/km²")
    iris_ok = True
except Exception as e:
    print("⚠️  Erreur lors du chargement/assemblage IRIS.")
    print(f"   Erreur : {e}")
    print("   → Relancez d'abord la cellule de diagnostic ci-dessus pour vérifier les noms de colonnes.")
    iris_ok = False


In [ ]:
# ── Rapprochement patient -> IRIS par CODE_IRIS (déjà présent dans le fichier patients) ──
if iris_ok:
    df = df.drop(columns=[c for c in ['densite_pop'] if c in df.columns])  # rejouable sans collision

    df['CODE_IRIS'] = df['CODE_IRIS'].astype(str).str.strip().str.zfill(9)

    df = df.merge(
        iris[[CONFIG['col_iris_code_shp'], 'densite_pop']].rename(
            columns={CONFIG['col_iris_code_shp']: 'CODE_IRIS'}
        ),
        on='CODE_IRIS', how='left'
    )

    n_manquants = df['densite_pop'].isna().sum()
    print(f"✅ Densité de population assignée à {len(df) - n_manquants}/{len(df)} patients (jointure par CODE_IRIS)")
    if n_manquants > 0:
        print(f"   ⚠️  {n_manquants} patients sans correspondance (format de code IRIS différent ?)")


In [ ]:
# ── Les patients NF habitent-ils des zones moins denses ? ───────────────────
if iris_ok:
    d_A = df.loc[df['groupe_A'] == 1, 'densite_pop'].dropna()
    d_B = df.loc[df['groupe_A'] == 0, 'densite_pop'].dropna()
    _, p_dens = mannwhitneyu(d_A, d_B, alternative='two-sided')

    print("── Densité de population : Groupe A (mutations NF) vs reste ──")
    print(f"  médiane NF   = {d_A.median():.0f} hab/km²")
    print(f"  médiane reste= {d_B.median():.0f} hab/km²")
    print(f"  p = {'<0.001' if p_dens < 0.001 else f'{p_dens:.3f}'} "
          f"{'✅ différence significative' if p_dens < CONFIG['alpha'] else '— pas de différence significative'}")

    fig, ax = plt.subplots(figsize=(5, 5))
    bp = ax.boxplot([d_A, d_B], labels=['Groupe A\n(mut. NF)', 'Reste'],
                     patch_artist=True, widths=0.5, showfliers=False)
    bp['boxes'][0].set_facecolor('#2E86AB'); bp['boxes'][0].set_alpha(0.7)
    bp['boxes'][1].set_facecolor('#AAAAAA'); bp['boxes'][1].set_alpha(0.7)
    ax.set_ylabel('Densité de population (hab/km²)')
    ax.set_title('Densité du quartier de résidence', fontweight='bold')
    ax.grid(True, axis='y', alpha=0.3)
    plt.tight_layout(); plt.show()


In [ ]:
# ── Moran bivarié SÉPARÉMENT par strate de densité (réutilise le test de la Partie 5) ──
if iris_ok:
    df['strate_densite'] = pd.qcut(df['densite_pop'], q=3, labels=['Rural/périurbain', 'Intermédiaire', 'Urbain dense'])
    print("Répartition des strates :")
    print(df['strate_densite'].value_counts())
    print()

    resultats_strates = []
    for strate in ['Rural/périurbain', 'Intermédiaire', 'Urbain dense']:
        sous_df = df[df['strate_densite'] == strate].reset_index(drop=True)
        if sous_df['groupe_A'].sum() < 10:
            print(f"⚠️  Strate '{strate}' : trop peu de patients mutés NF, test ignoré")
            continue
        W_strate, _ = moran.construire_matrice_poids(sous_df, rayon_m=CONFIG['rayon_m'],
                                                       n_permutations=CONFIG['n_permutations'])
        res_strate = moran.moran_bivarie(
            sous_df, W_strate,
            var_x='groupe_A', var_y='score_expo_SH',
            label_x='Mutations NF', label_y=f'Score expo SH ({strate})',
            n_perm=CONFIG['n_permutations']
        )
        resultats_strates.append({
            'strate': strate, 'n': len(sous_df), 'n_NF': sous_df['groupe_A'].sum(),
            'I': res_strate['I'], 'p_sim': res_strate['p_sim'],
            'sig': '✅' if res_strate['p_sim'] < CONFIG['alpha'] else '—'
        })

    df_strates = pd.DataFrame(resultats_strates)
    print("\n── Résultat du Moran bivarié (Partie 5), strate par strate ──")
    print(df_strates.to_string(index=False))


In [ ]:
# ── Régression logistique ajustée (vue individuelle, pas spatiale) ──────────
if iris_ok:
    import statsmodels.api as sm
    import statsmodels.formula.api as smf

    # Important : age_diagnostic doit être numérique, sinon le modèle le traite comme
    # une catégorie (une variable par âge !) et ne converge pas.
    df['age_diagnostic'] = pd.to_numeric(df['age_diagnostic'], errors='coerce')

    colonnes_modele = ['groupe_A', 'score_expo_SH', 'densite_pop', CONFIG['col_tabac']]
    for c in [CONFIG['col_age'], CONFIG['col_sexe']]:
        if c in df.columns:
            colonnes_modele.append(c)

    df_modele = df[colonnes_modele].dropna()
    formule = f"groupe_A ~ score_expo_SH + densite_pop + {CONFIG['col_tabac']}"
    if CONFIG['col_age'] in df_modele.columns:
        formule += f" + {CONFIG['col_age']}"
    if CONFIG['col_sexe'] in df_modele.columns:
        formule += f" + C({CONFIG['col_sexe']})"

    print(f"Modèle ajusté : {formule}\n")
    modele = smf.logit(formule, data=df_modele).fit(disp=0)
    print(modele.summary())


#### 📊 Résultats — Partie 8 (densité)

Densité médiane : NF = **13 106 hab/km²**, reste = **14 088 hab/km²** (p=0,456, pas de
différence globale). **Mais en testant séparément par strate de densité, le résultat
positif de la Partie 5 ne tient nulle part** : Rural/périurbain I=0,0105 (p=0,320, NS) ;
Intermédiaire I=0,0066 (p=0,286, NS) ; Urbain dense **I=-0,0123 (p=0,003, significatif
mais NÉGATIF)**. → Le signal global positif s'efface ou s'inverse une fois qu'on compare
des quartiers de densité comparable — signe d'un possible effet de confusion par
l'urbanité plutôt qu'un vrai lien spécifique à la mutation.

⚠️ La régression ajustée contenait un bug dans une version précédente (âge traité comme
catégorie, modèle non convergent) — **corrigé ci-dessus** (`pd.to_numeric` sur
`age_diagnostic`). Relancez la cellule pour obtenir un résultat fiable si vous modifiez
le modèle.

---
## PARTIE 9 — Robustesse : est-ce que ça dépend du rayon choisi (3 km) ?

**L'idée en clair :** la Partie 5 a tout calculé avec un rayon fixe de 3 km — un choix
arbitraire. Si le résultat n'existe qu'à exactement 3 km et disparaît à 2 km ou 5 km,
c'est mauvais signe : ça voudrait dire que c'est un hasard statistique lié à ce chiffre
précis, pas un vrai phénomène géographique.

**Ce qu'on fait ici :** on refait exactement le même test (Moran bivarié NF × Seveso,
Partie 5a) avec plusieurs rayons : 1, 2, 3, 5 et 10 km.

In [ ]:
resultats_rayons = []

for r in CONFIG['rayons_test_m']:
    print(f"→ Rayon {r/1000:.0f} km...")
    df_r = df.copy()
    df_r = moran.calculer_expositions_icpe(df_r, icpe_sh, icpe_sb, r)
    W_r, n_voisins_r = moran.construire_matrice_poids(df_r, rayon_m=r,
                                                       n_permutations=CONFIG['n_permutations'])
    res_r = moran.moran_bivarie(
        df_r, W_r,
        var_x='groupe_A', var_y='score_expo_SH',
        label_x='Mutations NF', label_y='Score expo SH',
        n_perm=CONFIG['n_permutations']
    )
    resultats_rayons.append({
        'rayon_km': r / 1000,
        'I': res_r['I'],
        'p_sim': res_r['p_sim'],
        'sig': '✅' if res_r['p_sim'] < CONFIG['alpha'] else '—',
        'isoles': sum(1 for n in n_voisins_r if n == 0)
    })

df_rayons = pd.DataFrame(resultats_rayons)
print("\n── Sensibilité au rayon (test de la Partie 5a, répété) ──")
print(df_rayons.to_string(index=False))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].plot(df_rayons['rayon_km'], df_rayons['I'], 'o-', color='#2E86AB', linewidth=2)
axes[0].axhline(0, color='grey', linestyle='--', alpha=0.5)
axes[0].set_xlabel('Rayon (km)'); axes[0].set_ylabel("Indice de Moran bivarié (I)")
axes[0].set_title("L'effet selon le rayon choisi", fontweight='bold')
axes[0].grid(True, alpha=0.3)

axes[1].plot(df_rayons['rayon_km'], df_rayons['p_sim'], 'o-', color='#E76F51', linewidth=2)
axes[1].axhline(CONFIG['alpha'], color='red', linestyle='--', label=f"seuil α={CONFIG['alpha']}")
axes[1].set_xlabel('Rayon (km)'); axes[1].set_ylabel('p-value (permutation)')
axes[1].set_title('Significativité selon le rayon', fontweight='bold')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout(); plt.show()


#### 📊 Résultats — Partie 9 (rayon)

| Rayon | I | p_sim | Sig. |
|---|---|---|---|
| 1 km | 0,0018 | 0,432 | — |
| 2 km | 0,0107 | 0,084 | — |
| **3 km** | **0,0164** | **0,030** | **✅** |
| 5 km | 0,0002 | 0,448 | — |
| 10 km | -0,0006 | 0,436 | — |

**Le résultat de la Partie 5 n'existe qu'à exactement 3 km** — avant et après, il
s'effondre à quasiment zéro. C'est le signe d'un coup de chance statistique lié à ce
choix précis de rayon, pas un vrai phénomène géographique qui se renforcerait
progressivement.

---
## PARTIE 10 — Robustesse : un phénomène diffus, ou très local ?

**L'idée en clair :** il n'y a que 37 sites Seveso SH en Île-de-France. Les 64 patients
"co-cluster" (Partie 6a) pourraient en réalité être tirés par un tout petit nombre de
sites, voire un seul. Si c'est le cas, le résultat n'est pas un phénomène général en
Île-de-France, mais une histoire très locale.

**Ce qu'on fait ici :** on réutilise directement `df['lisa_bv_type_A']` (le résultat de
la Partie 6a, préservé sous ce nom — voir le correctif qui y a été ajouté) et
`clusters_bv_A` déjà calculés en Partie 6a (aucun recalcul du LISA bivarié) pour
identifier, pour chacun des 64 patients HH, le site Seveso SH le plus proche.

In [ ]:
from scipy.spatial import cKDTree

# On réutilise df['lisa_bv_type_A'] déjà calculé (et préservé) en Partie 6a --
# pas de recalcul du LISA bivarié. ⚠️ Bien utiliser le suffixe _A : la colonne générique
# 'lisa_bv_type' a été écrasée par la Partie 6b (groupe_AC) et ne correspond plus à la 6a.
patients_hh = df[df['lisa_bv_type_A'] == 'HH'].copy()
print(f"Patients en co-cluster HH : {len(patients_hh)}")

# Coordonnées des sites Seveso SH (en Lambert93, comme les patients)
coords_sh = np.column_stack([icpe_sh.geometry.x, icpe_sh.geometry.y])
arbre = cKDTree(coords_sh)

coords_patients = patients_hh[['x_l93', 'y_l93']].values
dist_proche, idx_proche = arbre.query(coords_patients)

patients_hh['site_id_proche'] = idx_proche
patients_hh['dist_site_proche_m'] = dist_proche


In [ ]:
comptage_sites = patients_hh['site_id_proche'].value_counts().sort_values(ascending=False)
n_sites_distincts = len(comptage_sites)

print(f"Nombre de sites Seveso distincts concernés : {n_sites_distincts} (sur 37 au total)")
print()
print("── Répartition des 64 patients HH par site le plus proche ──")
print(comptage_sites)

part_top3 = comptage_sites.head(3).sum() / len(patients_hh) * 100
print(f"\n→ Les 3 sites les plus représentés concentrent {part_top3:.0f}% des patients HH")


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
comptage_sites.plot(kind='bar', ax=ax, color='#E76F51', alpha=0.8)
ax.set_xlabel('Identifiant du site Seveso SH le plus proche')
ax.set_ylabel('Nombre de patients HH rattachés')
ax.set_title('Concentration des co-clusters autour des sites Seveso', fontweight='bold')
ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout(); plt.show()


#### 📊 Résultats — Concentration géographique

**12 sites distincts sur 37** sont concernés, mais **les 3 sites les plus représentés
concentrent 73% des 64 patients HH** (dont 42% pour un seul site). → Le résultat global
n'est **pas un effet répandu en Île-de-France** : c'est surtout l'histoire de 2-3
quartiers industriels précis.

### 10b — Caractéristiques des 3 sites Seveso principaux

**L'idée en clair :** maintenant qu'on sait que 3 sites concentrent la majorité des
patients "à risque", autant les identifier précisément : où sont-ils, dans quel IRIS,
quelle densité de population autour ?

In [ ]:
print("Colonnes disponibles dans icpe_sh :")
print(icpe_sh.columns.tolist())
print()
print(icpe_sh.head(2))


In [ ]:
top3_ids = comptage_sites.head(3).index.tolist()
sites_top3 = icpe_sh.iloc[top3_ids].copy()
sites_top3['site_id'] = top3_ids

if 'iris_ok' in dir() and iris_ok:
    if sites_top3.crs != iris.crs:
        sites_top3 = sites_top3.to_crs(iris.crs)
    sites_top3_iris = gpd.sjoin(
        sites_top3,
        iris[[CONFIG['col_iris_code_shp'], 'densite_pop', 'geometry']],
        how='left', predicate='within'
    )
else:
    sites_top3_iris = sites_top3.copy()
    sites_top3_iris[CONFIG['col_iris_code_shp']] = np.nan
    sites_top3_iris['densite_pop'] = np.nan
    print("⚠️  Partie 8 (densité IRIS) non exécutée -- IRIS/densité des sites non disponibles")

dist_par_site = patients_hh.groupby('site_id_proche')['dist_site_proche_m'].median()

colonnes_interessantes = [c for c in [
    'nom', 'NOM', 'nom_etablissement', 'NOM_ETABLISSEMENT', 'raison_sociale', 'RAISON_SOCIALE',
    'adresse', 'ADRESSE', 'lib_adr', 'commune', 'COMMUNE', 'INSEE_COM', 'NOM_COM', 'code_postal',
    'activite', 'ACTIVITE', 'libelle_activite', 'lib_activ', 'regime', 'REGIME', 'seveso', 'SEVESO',
] if c in sites_top3_iris.columns]
print("Colonnes pertinentes détectées automatiquement :", colonnes_interessantes)
print()

fiches_sites = []
for sid in top3_ids:
    row = sites_top3_iris[sites_top3_iris['site_id'] == sid].iloc[0]
    fiche = {'site_id': sid, 'nb_patients_HH': int(comptage_sites[sid])}
    print(f"── Site #{sid} — {fiche['nb_patients_HH']} patients HH rattachés ──")
    for c in colonnes_interessantes:
        print(f"   {c} : {row[c]}")
        fiche[c] = row[c]
    code_iris_site = row.get(CONFIG['col_iris_code_shp'], np.nan)
    densite_site = row.get('densite_pop', np.nan)
    dist_mediane = dist_par_site.get(sid, np.nan)
    print(f"   CODE_IRIS du site : {code_iris_site}")
    print(f"   Densité du quartier : {densite_site:.0f} hab/km²" if pd.notna(densite_site) else "   Densité du quartier : n.d.")
    print(f"   Distance médiane des patients HH à ce site : {dist_mediane:.0f} m" if pd.notna(dist_mediane) else "   Distance médiane : n.d.")
    print()
    fiche['CODE_IRIS'] = code_iris_site
    fiche['densite_pop'] = densite_site
    fiche['dist_mediane_patients_HH_m'] = dist_mediane
    fiches_sites.append(fiche)

df_sites_top3 = pd.DataFrame(fiches_sites)


### 10c — Caractéristiques des patients du cluster HH

**L'idée en clair :** qui sont, concrètement, les 64 patients identifiés comme étant en
zone de co-cluster ? Ce profil aide à juger si le signal est crédible et où concentrer
une éventuelle enquête de terrain.

In [ ]:
print("═"*70)
print(f"CARACTÉRISTIQUES DES {len(patients_hh)} PATIENTS DU CO-CLUSTER HH")
print("═"*70)

age_hh = pd.to_numeric(patients_hh['age_diagnostic'], errors='coerce')
print(f"\n— Âge au diagnostic —")
print(f"   Moyenne : {age_hh.mean():.1f} ans | Médiane : {age_hh.median():.0f} ans "
      f"| Min-Max : {age_hh.min():.0f}-{age_hh.max():.0f} ans")

print(f"\n— Sexe —")
print(patients_hh['sexe'].value_counts())

print(f"\n— Statut tabagique —")
print(patients_hh['statut_tabagique'].value_counts())

print(f"\n— Mutations présentes parmi ces patients —")
for m in CONFIG['mutations_nf']:
    col = 'mutation_' + m
    if col in patients_hh.columns:
        n = int(colonne_mutation_en_binaire(patients_hh[col]).sum())
        if n > 0:
            print(f"   {m:8s} : {n} patients ({n/len(patients_hh)*100:.0f}%)")

print(f"\n— Communes de résidence les plus représentées —")
if 'NOM_COM' in patients_hh.columns:
    print(patients_hh['NOM_COM'].value_counts().head(8))

print(f"\n— IRIS de résidence les plus représentés —")
if {'CODE_IRIS', 'NOM_IRIS'}.issubset(patients_hh.columns):
    print(patients_hh[['CODE_IRIS', 'NOM_IRIS']].value_counts().head(8))

print(f"\n— Exposition Seveso —")
print(f"   Distance médiane au site SH le plus proche : {patients_hh['dist_site_proche_m'].median():.0f} m")
print(f"   Score d'exposition SH médian : {patients_hh['score_expo_SH'].median():.4f}")
if 'densite_pop' in patients_hh.columns and patients_hh['densite_pop'].notna().any():
    print(f"   Densité médiane du quartier : {patients_hh['densite_pop'].median():.0f} hab/km²")


#### 📊 Résultats — Partie 10b/10c (sites & patients HH)

*Analyse ajoutée au notebook — à compléter après exécution (fiches des 3 sites et
profil détaillé des 64 patients HH, pas encore générés sur vos données réelles).*
- Identité/activité des 3 sites principaux : …
- Profil type des patients HH (âge, sexe, tabac, mutations dominantes) : …
- Communes/IRIS où concentrer une éventuelle enquête de terrain : …

---
## PARTIE 11 — Robustesse : a-t-on testé trop de choses en même temps ?

**L'idée en clair :** entre les Parties 3 et 5, on a fait 7 tests de Moran différents.
Plus on fait de tests, plus le risque de trouver un résultat "significatif" par pur
hasard augmente — un peu comme lancer une pièce 7 fois et s'étonner d'avoir eu "face"
au moins une fois.

**Ce qu'on fait ici :** on réutilise directement les 7 résultats déjà calculés en
Parties 3 et 5 (`res_moran_*`, `res_bv_*`) — **aucun recalcul** — et on leur applique
deux corrections statistiques classiques (Bonferroni, stricte ; et Benjamini-Hochberg/FDR,
plus souple).

In [ ]:
from statsmodels.stats.multitest import multipletests

# On réutilise les résultats déjà calculés en Parties 3 et 5 -- pas de recalcul
tests = [
    ("Moran global — Mutations NF (3a)",                    res_moran_A['p_sim']),
    ("Moran global — Non-fumeurs (3b)",                      res_moran_C['p_sim']),
    ("Moran global — Score expo Seveso SH (3c)",             res_moran_SH['p_sim']),
    ("Moran global — Nb Seveso SH 3km (3d)",                 res_moran_nb['p_sim']),
    ("Moran bivarié — NF × Score expo SH (5a)",              res_bv_A_score['p_sim']),
    ("Moran bivarié — NF × Nb SH 3km (5b)",                  res_bv_A_nb['p_sim']),
    ("Moran bivarié — (NF+non-fum) × Score expo SH (5c)",    res_bv_AC_score['p_sim']),
]

labels = [t[0] for t in tests]
p_bruts = [t[1] for t in tests]

sig_bonf, p_bonf, _, _ = multipletests(p_bruts, alpha=CONFIG['alpha'], method='bonferroni')
sig_fdr,  p_fdr,  _, _ = multipletests(p_bruts, alpha=CONFIG['alpha'], method='fdr_bh')

df_correction = pd.DataFrame({
    'Test': labels,
    'p brut': p_bruts,
    'Sig. brut (α=0.05)': ['✅' if p < CONFIG['alpha'] else '—' for p in p_bruts],
    'p Bonferroni': p_bonf.round(4),
    'Sig. Bonferroni': ['✅' if s else '—' for s in sig_bonf],
    'p FDR (Benjamini-Hochberg)': p_fdr.round(4),
    'Sig. FDR': ['✅' if s else '—' for s in sig_fdr],
})

print("── Effet de la correction pour tests multiples ──\n")
print(df_correction.to_string(index=False))


#### 📊 Résultats — Partie 11 (tests multiples)

Avec la correction de Bonferroni (la plus stricte) : seuls les tests sur le clustering
propre des sites Seveso (3c, 3d — attendu) et le test bivarié **NF × Nb_SH_3km (5b,
p=0,004 → reste significatif)** survivent. **Le test bivarié NF × Score_expo_SH (5a,
p=0,023) ne survit pas à Bonferroni**, mais reste significatif sous FDR, plus souple.
→ Un seul des deux tests bivariés clés (5b) est pleinement robuste à une correction
stricte.

---
## PARTIE 12 — Robustesse : le signal tient-il pour chaque mutation prise isolément ?

**L'idée en clair :** le "Groupe A" mélange 7 mutations. Si le signal de la Partie 5
est biologiquement réel, il devrait apparaître, au moins un peu, pour chacune. On les
teste séparément.

In [ ]:
colonnes_a_tester = ['mutation_' + m for m in CONFIG['mutations_nf']]

print("Exemple de valeurs brutes dans 'mutation_EGFR' :", df['mutation_EGFR'].unique()[:8])
print()

resultats_robustesse = []

for col in colonnes_a_tester:
    if col not in df.columns:
        print(f"⚠️  Colonne '{col}' absente de df — test ignoré")
        continue

    col_bin = col + '_bin'
    df[col_bin] = colonne_mutation_en_binaire(df[col])
    n_pos = int(df[col_bin].sum())

    if n_pos < 10:
        print(f"⚠️  '{col}' : seulement {n_pos} patients — trop peu pour un test fiable, ignoré")
        continue

    res_col = moran.moran_bivarie(
        df, W, var_x=col_bin, var_y='score_expo_SH',
        label_x=col, label_y='Score expo Seveso SH',
        n_perm=CONFIG['n_permutations']
    )
    resultats_robustesse.append({
        'Mutation testée': col, 'n patients': n_pos,
        'I': res_col['I'], 'p_sim': res_col['p_sim'],
        'Sig.': '✅' if res_col['p_sim'] < CONFIG['alpha'] else '—'
    })

# Pour comparaison, les deux groupes déjà testés en Partie 5 (réutilisés, pas recalculés)
resultats_robustesse.append({
    'Mutation testée': 'Groupe A (7 mutations, Partie 5a)', 'n patients': int(df['groupe_A'].sum()),
    'I': res_bv_A_score['I'], 'p_sim': res_bv_A_score['p_sim'],
    'Sig.': '✅' if res_bv_A_score['p_sim'] < CONFIG['alpha'] else '—'
})
resultats_robustesse.append({
    'Mutation testée': 'Groupe A+C (Partie 5c)', 'n patients': int(df['groupe_AC'].sum()),
    'I': res_bv_AC_score['I'], 'p_sim': res_bv_AC_score['p_sim'],
    'Sig.': '✅' if res_bv_AC_score['p_sim'] < CONFIG['alpha'] else '—'
})

df_robustesse = pd.DataFrame(resultats_robustesse)
print("── Le signal Seveso × mutation, mutation par mutation ──\n")
print(df_robustesse.to_string(index=False))


#### 📊 Résultats — Partie 12 (mutation par mutation)

| Mutation | n | I | p_sim | Sig. |
|---|---|---|---|---|
| EGFR | 178 | -0,0093 | 0,092 | — |
| ALK | 54 | -0,0134 | 0,010 | ✅ (négatif !) |
| ROS1 | 30 | 0,0488 | 0,001 | ✅ |
| ERBB2 | 32 | 0,0317 | 0,014 | ✅ |
| MET | 58 | 0,0213 | 0,013 | ✅ |

**EGFR (n=178, la mutation la plus fréquente du Groupe A) ne montre aucun signal, et
ALK (n=54) va même dans le sens opposé.** Le signal positif de la Partie 5 n'est donc
**pas partagé par les deux mutations majoritaires** (EGFR+ALK = 70% du Groupe A) — il
est porté uniquement par les mutations minoritaires ROS1/ERBB2/MET. À creuser
formellement en Partie 13.

---
## PARTIE 13 — Évaluation groupée : ROS1/ERBB2/MET vs EGFR/ALK

**L'idée en clair :** la Partie 12 a montré que ROS1, ERBB2 et MET vont dans un sens, et
qu'EGFR/ALK vont dans l'autre (ou nul). On regroupe maintenant chaque côté en un seul
groupe pour comparer plus formellement :
1. leur signal spatial respectif (Moran bivarié), et
2. si les deux groupes de patients se ressemblent par ailleurs (âge, sexe, tabac,
   densité du quartier...) — une différence démographique pourrait expliquer une partie
   de l'écart géographique observé, plutôt qu'un vrai effet spécifique à la mutation.

In [ ]:
df['groupe_positif'] = (
    colonne_mutation_en_binaire(df['mutation_ROS1']) |
    colonne_mutation_en_binaire(df['mutation_ERBB2']) |
    colonne_mutation_en_binaire(df['mutation_MET'])
).astype(int)

df['groupe_negatif'] = (
    colonne_mutation_en_binaire(df['mutation_EGFR']) |
    colonne_mutation_en_binaire(df['mutation_ALK'])
).astype(int)

chevauchement = int(((df['groupe_positif'] == 1) & (df['groupe_negatif'] == 1)).sum())

print(f"Groupe positif (ROS1/ERBB2/MET) : {int(df['groupe_positif'].sum())} patients")
print(f"Groupe négatif (EGFR/ALK)       : {int(df['groupe_negatif'].sum())} patients")
print(f"Chevauchement (mutation des deux côtés) : {chevauchement} patients")
if chevauchement > 0:
    print("  -> ces patients comptent dans les deux groupes ci-dessous (à garder en tête,")
    print("     ex. amplification MET secondaire chez un patient EGFR+ déjà traité)")


In [ ]:
res_positif = moran.moran_bivarie(
    df, W, var_x='groupe_positif', var_y='score_expo_SH',
    label_x='Groupe positif (ROS1/ERBB2/MET)', label_y='Score expo Seveso SH',
    n_perm=CONFIG['n_permutations']
)
res_negatif = moran.moran_bivarie(
    df, W, var_x='groupe_negatif', var_y='score_expo_SH',
    label_x='Groupe négatif (EGFR/ALK)', label_y='Score expo Seveso SH',
    n_perm=CONFIG['n_permutations']
)

print("── Signal spatial : groupe positif vs groupe négatif ──\n")
print(f"{'Groupe':32s} {'n':>5s} {'I':>9s} {'p_sim':>8s}  Sig.")
for label, n_grp, res in [
    ('Positif (ROS1/ERBB2/MET)', int(df['groupe_positif'].sum()), res_positif),
    ('Négatif (EGFR/ALK)',       int(df['groupe_negatif'].sum()), res_negatif),
]:
    sig = '✅' if res['p_sim'] < CONFIG['alpha'] else '—'
    print(f"{label:32s} {n_grp:5d} {res['I']:9.4f} {res['p_sim']:8.3f}  {sig}")


In [ ]:
from scipy.stats import chi2_contingency

grp_pos = df[df['groupe_positif'] == 1]
grp_neg = df[df['groupe_negatif'] == 1]
print(f"Comparaison : groupe positif (n={len(grp_pos)}) vs groupe négatif (n={len(grp_neg)})\n")

age_pos = pd.to_numeric(grp_pos['age_diagnostic'], errors='coerce').dropna()
age_neg = pd.to_numeric(grp_neg['age_diagnostic'], errors='coerce').dropna()
_, p_age = mannwhitneyu(age_pos, age_neg, alternative='two-sided')
print(f"Âge médian            -- positif : {age_pos.median():.0f} ans | négatif : {age_neg.median():.0f} ans | p = {p_age:.3f}")

comp_sexe = pd.concat([
    grp_pos[['sexe']].assign(groupe='Positif'),
    grp_neg[['sexe']].assign(groupe='Négatif'),
])
tab_sexe = pd.crosstab(comp_sexe['sexe'], comp_sexe['groupe'])
chi2, p_sexe, _, _ = chi2_contingency(tab_sexe)
print(f"\nSexe -- p = {p_sexe:.3f}")
print(tab_sexe)

comp_tabac = pd.concat([
    grp_pos[['statut_tabagique']].assign(groupe='Positif'),
    grp_neg[['statut_tabagique']].assign(groupe='Négatif'),
])
tab_tabac = pd.crosstab(comp_tabac['statut_tabagique'], comp_tabac['groupe'])
chi2, p_tabac, _, _ = chi2_contingency(tab_tabac)
print(f"\nStatut tabagique -- p = {p_tabac:.3f}")
print(tab_tabac)

dist_pos = grp_pos['dist_SH_m'].dropna()
dist_neg = grp_neg['dist_SH_m'].dropna()
_, p_dist = mannwhitneyu(dist_pos, dist_neg, alternative='two-sided')
print(f"\nDistance médiane au site SH le + proche -- positif : {dist_pos.median():.0f} m | négatif : {dist_neg.median():.0f} m | p = {p_dist:.3f}")

score_pos = grp_pos['score_expo_SH'].dropna()
score_neg = grp_neg['score_expo_SH'].dropna()
_, p_score = mannwhitneyu(score_pos, score_neg, alternative='two-sided')
print(f"Score expo SH médian                    -- positif : {score_pos.median():.4f} | négatif : {score_neg.median():.4f} | p = {p_score:.3f}")

if 'densite_pop' in df.columns and df['densite_pop'].notna().any():
    dens_pos = grp_pos['densite_pop'].dropna()
    dens_neg = grp_neg['densite_pop'].dropna()
    _, p_dens = mannwhitneyu(dens_pos, dens_neg, alternative='two-sided')
    print(f"Densité médiane du quartier              -- positif : {dens_pos.median():.0f} hab/km² | négatif : {dens_neg.median():.0f} hab/km² | p = {p_dens:.3f}")
else:
    print("(densité non disponible -- exécutez la Partie 8 pour l'inclure dans cette comparaison)")


#### 📊 Résultats — Partie 13

*Analyse ajoutée au notebook — à compléter après exécution (pas encore lancée sur vos
données réelles).*
- Le groupe positif reste-t-il significatif une fois regroupé ? Le négatif reste-t-il nul ? …
- Les deux groupes diffèrent-ils par âge/sexe/tabac/densité (confondants possibles) ? …
- Conclusion : la différence ROS1/ERBB2/MET vs EGFR/ALK est-elle purement géographique,
  ou en partie démographique ? …

---
## PARTIE 14 — Cohérence avec l'analyse individuelle (régression logistique)

**L'idée en clair :** ce notebook regarde la géographie (Moran). Mais une autre analyse
du projet (régression logistique, hors de ce notebook) a déjà regardé, patient par
patient, si les patients EGFR/NF habitent plus souvent près des sites ICPE-SH (sans
notion spatiale de "clustering", juste "proche ou pas"). Si les deux méthodes,
complètement différentes, vont dans le même sens, c'est un signal plus crédible que
chacune isolément.

#### 📊 Comparaison — tableau à tenir à jour une fois les deux analyses finalisées

| | Régression logistique (individuelle) | Moran bivarié (spatial, ce notebook) |
|---|---|---|
| Sens de l'association | Plus proche des ICPE-SH chez NF (OR≈1,16, p=0,052, à la limite) | Co-clustering positif (I=0,016-0,023, p=0,004-0,023) |
| Force du signal | Faible / à la limite | Faible, fragile (Parties 8-12) |
| Robuste aux contrôles ? | À vérifier (ajustement âge/sexe/tabac déjà fait ?) | Non pleinement (densité, rayon — Parties 8-9) |
| **Conclusion croisée** | Deux signaux faibles, dans le même sens — convergence à confirmer, pas une preuve | |

---
## PARTIE 15 — Synthèse finale globale

In [ ]:
print("═"*70)
print("SYNTHÈSE FINALE — ANALYSE PRINCIPALE + ROBUSTESSE")
print("═"*70)

print('''
ANALYSE PRINCIPALE (Parties 0-7)
   1. Mutations NF : pas de clustering propre (I=-0,008, p=0,224)
   2. Score expo Seveso SH : fort clustering propre, attendu (I=0,656, p=0,001)
   3. Moran bivarié NF x Score expo SH : I=0,016, p=0,023 -- SIGNIFICATIF mais FAIBLE
   4. 64 patients en co-cluster HH

TESTS DE ROBUSTESSE (Parties 8-13)
   8.  Densité (ville/campagne)     -> le signal s'efface/s'inverse par strate  [FRAGILE]
   9.  Sensibilité au rayon          -> significatif à 3km SEULEMENT            [NE TIENT PAS]
   10. Concentration géographique    -> 73%% des patients HH sur 3 sites         [FRAGILE/LOCAL]
   11. Tests multiples               -> 1 test sur 2 survit à Bonferroni        [FRAGILE]
   12. Mutation par mutation         -> EGFR/ALK (70%% du groupe) ne suivent pas [NE TIENT PAS]
   13. ROS1/ERBB2/MET vs EGFR/ALK    -> à exécuter sur les données réelles

──────────────────────────────────────────────────────────────────
VERDICT GLOBAL
──────────────────────────────────────────────────────────────────
Le signal positif détecté en Partie 5 (I=0,016, p=0,023) est statistiquement réel
au sens du test, mais NE RÉSISTE PAS BIEN à l'examen : il dépend d'un rayon précis,
se concentre sur 2-3 sites industriels, et n'est pas partagé par les deux mutations
NF les plus fréquentes (EGFR, ALK). Le sous-groupe ROS1/ERBB2/MET montre en revanche
un signal plus cohérent sur plusieurs vérifications -- une piste à creuser
spécifiquement, plutôt qu'une conclusion sur les mutations NF prises comme un bloc.

Rappel méthodologique : même un résultat robuste resterait une CORRÉLATION spatiale,
pas une preuve qu'habiter près d'un site Seveso CAUSE une mutation. Des facteurs de
confusion géographiques (densité, urbanité) doivent être exclus avant toute conclusion.
''')


---
**Notebook combiné — fin.** Analyse principale (Parties 0-7) et tests de robustesse
(Parties 8-15) partagent un seul jeu de données, une seule matrice de poids spatiaux,
et aucun calcul de Moran n'est dupliqué entre les deux parties.